In [ ]:
# ── Imports standard ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')
import os

os.chdir("/home/onyxia/work/PESSD") 



In [3]:
PATH_DECES = 'data/Deces_par_age.xlsx'   
PATH_POP   = 'data/Pop+65ans_fr.xlsx'

def charger_fichier(path, label):
    if path is None or not os.path.exists(path):
        if IN_COLAB:
            print(f'📂 Upload : {label}')
            uploaded = files.upload()
            return pd.read_excel(list(uploaded.keys())[0], header=0)
        raise FileNotFoundError(f'Introuvable : {path}')
    return pd.read_excel(path, header=0)

df_deces_raw = charger_fichier(PATH_DECES, 'Deces_par_age.xlsx')
df_pop_raw   = charger_fichier(PATH_POP,   'Pop_65ans_fr.xlsx')

In [12]:
# ── Décès : wide (700 colonnes = 20 ans × 35 ages) → long ────────────────────
ages_labels = df_deces_raw.iloc[0, 1:].values
col_names   = df_deces_raw.columns[1:].tolist()

annees_col, ages_col = [], []
for col, age_label in zip(col_names, ages_labels):
    annees_col.append(int(str(col).split('.')[0]))
    ages_col.append(int(str(age_label).replace(' ans', '').strip()))

df_data = df_deces_raw.iloc[1:].rename(columns={'TIME': 'Pays'}).reset_index(drop=True)
PAYS_LIST = df_data['Pays'].tolist()

records = []
for _, row in df_data.iterrows():
    pays = row['Pays']
    for col, annee, age in zip(col_names, annees_col, ages_col):
        try:
            val = float(str(row[col]).replace(' ', '').replace(':', 'nan'))
        except:
            val = np.nan
        records.append({'Pays': pays, 'Annee': annee, 'Age': age, 'Deces': val})

df_deces = pd.DataFrame(records)

# ── Population 65+ : wide → long ────────────────────────────────────────────
df_pop = df_pop_raw.rename(columns={'TIME': 'Pays'})
annee_cols = [c for c in df_pop.columns if c != 'Pays']
df_pop_long = df_pop.melt(id_vars='Pays', value_vars=annee_cols,
                           var_name='Annee', value_name='Pop65plus')
df_pop_long['Annee']    = df_pop_long['Annee'].astype(int)
df_pop_long['Pop65plus'] = pd.to_numeric(
    df_pop_long['Pop65plus'].astype(str).str.replace(':', 'nan').str.strip(),
    errors='coerce'
)

In [13]:
# ── Cellule 4 : Construction variable Décès sous 2 ans — TOUS PAYS ───────────
# Paramètres définis ici pour éviter toute dépendance entre cellules
ANNEE_DEBUT = 2004
ANNEE_FIN   = 2023   # 2023 sera NaN (D_{t+1} indisponible) → série effective 2004–2022

all_results = []

for pays in PAYS_LIST:

    # Filtrage décès pays (on prend jusqu'à ANNEE_FIN+1 pour D_{t+1})
    d = df_deces[
        (df_deces['Pays']  == pays) &
        (df_deces['Annee'] >= ANNEE_DEBUT) &
        (df_deces['Annee'] <= ANNEE_FIN + 1)
    ]

    # Filtrage population pays
    p = df_pop_long[
        (df_pop_long['Pays']  == pays) &
        (df_pop_long['Annee'] >= ANNEE_DEBUT) &
        (df_pop_long['Annee'] <= ANNEE_FIN)
    ]

    # Composante 1 : D_t(≥65) — décès cette année parmi les 65+
    D_t = (
        d[d['Age'] >= 65]
        .groupby('Annee')['Deces'].sum()
        .rename('D_t_65plus')
    )

    # Composante 2 : D_{t+1}(≥66) — décès l'année suivante parmi les 66+
    # On décale l'index de -1 pour ramener sur l'année t
    D_t1 = (
        d[d['Age'] >= 66]
        .groupby('Annee')['Deces'].sum()
        .rename('D_t1_66plus')
    )
    D_t1.index = D_t1.index - 1

    # Assemblage
    df_r = (
        pd.DataFrame({'D_t_65plus': D_t, 'D_t1_66plus': D_t1})
        .loc[ANNEE_DEBUT:ANNEE_FIN]
        .join(p.set_index('Annee')['Pop65plus'])
    )
    df_r['Deces_sous_2ans'] = (df_r['D_t_65plus'] + df_r['D_t1_66plus']) / df_r['Pop65plus']
    df_r['Pays'] = pays
    df_r.index.name = 'Annee'
    all_results.append(df_r.reset_index())

df_all = pd.concat(all_results, ignore_index=True)
valid  = df_all.dropna(subset=['Deces_sous_2ans'])

In [11]:
# ── Cellule 5 : Tableau pivot lisible ────────────────────────────────────────

df_pivot = valid.pivot(index='Pays', columns='Annee', values='Deces_sous_2ans') * 100
df_pivot.columns.name = None
df_pivot = df_pivot.round(2)

df_pivot['Moyenne']   = df_pivot.mean(axis=1).round(2)
df_pivot['Var_2004_2022'] = ((df_pivot[2022] - df_pivot[2004]) / df_pivot[2004] * 100).round(1)
df_pivot = df_pivot.sort_values('Moyenne', ascending=False)

display(
    df_pivot.style
    .background_gradient(subset=list(range(2004, 2023)), cmap='YlOrRd', axis=None)
    .background_gradient(subset=['Moyenne'], cmap='Blues')
    .applymap(
        lambda v: 'color:green;font-weight:bold' if isinstance(v, float) and v < 0
             else ('color:red;font-weight:bold'  if isinstance(v, float) and v > 0 else ''),
        subset=['Var_2004_2022']
    )
    .format('{:.2f}', subset=list(range(2004, 2023)))
    .format('{:.2f}', subset=['Moyenne'])
    .format('{:+.1f}%', subset=['Var_2004_2022'])
    .set_caption('Décès sous 2 ans (% pop. 65+) — Vert = baisse | Rouge = hausse (2004→2022)')
)

df_pivot_export = valid.pivot(index='Pays', columns='Annee', values='Deces_sous_2ans').mul(100).round(3)
df_pivot_export.to_excel('data/deces_sous_2ans_taux.xlsx')

,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,Moyenne,Var_2004_2022
Pays,,,,,,,,,,,,,,,,,,,,,
Lettonie,11.78,11.77,11.77,11.51,11.12,11.12,11.00,10.99,11.11,11.02,11.02,11.06,11.15,11.21,11.06,11.17,12.43,12.67,11.43,11.39,-3.0%
Hongrie,11.89,11.79,11.57,11.47,11.30,11.34,11.21,11.19,11.18,10.96,11.09,11.03,10.96,11.04,10.92,11.27,12.05,11.75,10.74,11.30,-9.7%
Lituanie,10.61,10.84,10.91,10.81,10.63,10.67,10.69,10.62,10.82,10.85,11.04,11.17,11.11,11.05,10.74,11.25,12.51,12.44,10.92,11.04,+2.9%
Slovaquie,11.87,11.92,11.79,11.59,11.34,11.32,11.08,10.84,10.74,10.37,10.26,10.12,9.90,9.77,9.41,9.66,11.11,10.90,9.22,10.70,-22.3%
Estonie,10.83,10.65,10.59,10.41,10.12,10.06,9.87,9.75,9.80,9.79,9.70,9.58,9.63,9.71,9.59,9.48,10.34,10.69,9.80,10.02,-9.5%
Pologne,10.18,10.10,10.07,10.12,10.22,10.22,10.05,10.11,10.13,9.84,9.74,9.64,9.53,9.62,9.46,10.02,11.07,10.66,9.34,10.01,-8.3%
Tchéquie,11.20,10.97,10.61,10.45,10.45,10.35,10.09,10.03,9.89,9.52,9.42,9.32,9.14,9.16,9.02,9.58,10.47,10.02,8.92,9.93,-20.4%
Danemark,10.78,10.67,10.54,10.30,10.07,9.84,9.40,9.00,8.75,8.42,8.24,8.21,8.14,8.24,8.17,8.02,8.17,8.44,8.43,9.04,-21.8%
Grèce,8.77,8.67,8.77,8.79,8.66,8.62,8.65,8.86,8.79,8.58,8.88,8.97,9.03,9.01,8.96,9.30,9.86,10.14,9.70,9.00,+10.6%
